# ⚖️ Indian Legal Judgment Prediction & 4-Tier Memory Engine
### Dual-Stage AI Pipeline: `InLegalBERT` + `Nyaya-Llama-3.1-8B` (4-Bit) + Statutory Concordance (BNS/BNSS)

**Architecture Highlights:**
- **Stage 1 Statistical Classifier:** `law-ai/InLegalBERT` (Fine-tuned on ILDC for Petition Allowed vs Dismissed prediction)
- **Stage 2 Judicial Reasoning:** `Raazi29/Nyaya-Llama-3.1-8B-Indian-Legal` (Quantized to 4-bit via BitsAndBytes to fit within ~5.0 GB VRAM)
- **4-Tier Agent Memory (TencentDB Model):** L0 (Raw text) -> L1 (Atomic entities) -> L2 (Procedural timeline) -> L3 (Persona & Posture)
- **Statutory Concordance Engine:** Bi-directional mapping between pre-2024 laws (IPC, CrPC, IEA) and post-July 2024 laws (BNS, BNSS, BSA)
- **Colab Free Tier Optimized:** Total VRAM usage ~8.4 GB on Colab T4 GPU (leaves >6 GB headroom to avoid OOM crashes)

## 1. Verify GPU Hardware (T4 GPU / 15 GB VRAM)

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Persistent Checkpointing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Indian_Legal_Judgment_Engine', exist_ok=True)
os.makedirs('/content/legal_runtime', exist_ok=True)
print("✅ Runtime directories initialized.")

## 3. Install Required Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 bitsandbytes>=0.43.0
!pip install -q chromadb sentence-transformers gradio pydantic tabulate scikit-learn
print("✅ Dependencies successfully installed.")

## 3.1 Unpack Project Codebase
Colab will automatically look for `muhsina_judgement_prediction.zip` in `/content` (drag-and-drop into Colab Files tab) or in your Google Drive.

In [ ]:
import os, zipfile

if not os.path.exists("/content/pipeline.py"):
    zip_candidates = [
        "/content/muhsina_judgement_prediction.zip",
        "/content/drive/MyDrive/muhsina_judgement_prediction.zip",
        "/content/drive/MyDrive/Indian_Legal_Judgment_Engine/muhsina_judgement_prediction.zip"
    ]
    found_zip = next((loc for loc in zip_candidates if os.path.exists(loc)), None)
    if found_zip:
        print(f"📦 Unpacking project archive from {found_zip}...")
        with zipfile.ZipFile(found_zip, "r") as z:
            z.extractall("/content")
        print("✅ Project files successfully extracted to /content!")
    else:
        print("💡 Tip: Drag and drop muhsina_judgement_prediction.zip into the left Files tab in Colab, then re-run this cell.")
else:
    print("✅ Project files already present in /content.")

## 4. Initialize Pipeline & Load Pre-Trained Legal Models

In [ ]:
import sys
# Ensure project files are on Python path
if '.' not in sys.path:
    sys.path.append('.')

from pipeline import LegalJudgmentPipeline

# Initialize pipeline (set load_gpu_models=True on Colab with T4 GPU)
pipeline = LegalJudgmentPipeline(load_gpu_models=True)
print("✅ Legal Judgment Pipeline initialized.")

## 5. Execute Sample Adjudication (Regular Bail u/s 439 CrPC / 483 BNSS)

In [ ]:
sample_petition = """
IN THE HIGH COURT OF DELHI AT NEW DELHI
BAIL APPLN. NO. 1420 OF 2024

IN THE MATTER OF:
Mohd. Tariq ... Petitioner / Accused
VERSUS
State (NCT of Delhi) ... Respondent

FIR No.: 342/2023
Police Station: Connaught Place, New Delhi
Under Sections: 420, 467, 468, 471, 120B of the Indian Penal Code, 1860
[Corresponding Provisions: Sections 318(4), 338, 336(3), 340(2), 61(2) of Bharatiya Nyaya Sanhita, 2023]

APPLICATION UNDER SECTION 439 OF THE CODE OF CRIMINAL PROCEDURE, 1973
FOR GRANT OF REGULAR BAIL

MOST RESPECTFULLY SHOWETH:
1. The Petitioner has been in continuous judicial custody since 14.10.2023 (exceeding 165 days).
2. The allegations in FIR No. 342/2023 registered at PS Connaught Place pertain to alleged financial irregularities and forgery of commercial purchase orders.
3. The investigating officer has already concluded the investigation and filed the comprehensive Final Report / Chargesheet under Section 173 CrPC before the Learned Chief Metropolitan Magistrate on 10.01.2024.
4. All relevant computers, ledgers, and original documents have been formally seized by the police. No recovery remains to be effected from the applicant.
5. The Petitioner has no previous criminal antecedents of any nature whatsoever.
6. Continued detention amounts to pre-trial punishment, violating Article 21 (Satender Kumar Antil v. CBI; Sanjay Chandra v. CBI).
"""

result = pipeline.run(sample_petition, case_id="COLAB_TEST_001")

print("\n--- STAGE 1 PREDICTION ---")
print(f"Outcome: {result['stage1']['outcome']}")
print(f"Allowed Probability: {result['stage1']['probability_allowed']:.1%}")
print(f"Dismissed Probability: {result['stage1']['probability_dismissed']:.1%}")

print("\n--- TIER L1 ATOMIC FACTS ---")
for k, v in result['L1_facts'].items():
    print(f"{k}: {v}")

print("\n--- STAGE 2 RATIO DECIDENDI ---")
print(result['stage2']['ratio_decidendi'])

## 6. Launch Gradio Dashboard with Public URL

In [ ]:
from app import demo
# share=True generates a public https://xxxx.gradio.live URL for free sharing!
demo.launch(share=True)

## 7. Backup Memory Snapshot to Google Drive

In [ ]:
status = pipeline.backup_session()
print("✅ Backup Completed Successfully:")
print(f"Saved to: {status['backup_file']}")
print(f"Latest snapshot: {status['latest_file']}")